# VMD Denoising Pipeline

This notebook demonstrates a **Variational Mode Decomposition (VMD)** denoising
pipeline applied to Project 8 I/Q time-series data.

**Pipeline overview:**
1. Load a batch of clean I/Q signals from HDF5 data files
2. Add nominal Gaussian noise (`noise_const = 1.0`) using the project's noise model
3. Decompose each noisy channel into **K = 3** VMD modes
4. Reconstruct the denoised signal by summing all modes (VMD acts as an adaptive
   band-pass filter bank, suppressing out-of-band noise)
5. Evaluate denoising quality: time-domain overlays, PSDs, and SNR metrics

## 0. Install / verify `vmdpy`

`vmdpy` is a lightweight pure-Python implementation of the VMD algorithm
(Dragomiretskiy & Zosso, 2014). Install it if it is not already present.

In [ ]:
try:
    import vmdpy
    print('vmdpy already installed.')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'vmdpy', '-q'])
    import vmdpy
    print('vmdpy installed successfully.')

## 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import h5py
from scipy import signal as sp_signal
from vmdpy import VMD

from src.utils.noise import noise_model

print('All imports successful.')

## 2. Configuration

Update `DATA_DIR` to point to the directory containing your HDF5 files.

In [ ]:
# ── Update this path ──────────────────────────────────────────────────────────
DATA_DIR = '/path/to/data/'   # directory containing .hdf5 / .h5 files
# ─────────────────────────────────────────────────────────────────────────────

INPUTS       = ['output_ts_I', 'output_ts_Q']   # HDF5 dataset keys
CUTOFF       = 4000                              # samples to keep per trace
NOISE_CONST  = 1.0                               # nominal noise level
N_EXAMPLES   = 4                                 # signals to visualise
SAMPLING_FREQ = 403e6                            # Hz (from noise model)

# ── VMD hyper-parameters ─────────────────────────────────────────────────────
K     = 3       # number of modes
ALPHA = 2000    # bandwidth constraint (moderate = well-separated modes)
TAU   = 0.0     # noise tolerance (0 = strict)
DC    = 0       # do not force a DC mode
INIT  = 1       # initialise mode frequencies uniformly across [0, f_s/2]
TOL   = 1e-7    # convergence tolerance
# ─────────────────────────────────────────────────────────────────────────────

CHANNEL_LABELS = ['I channel', 'Q channel']

## 3. Load clean signals from HDF5

In [ ]:
hdf5_files = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith('.hdf5') or f.endswith('.h5')
])
if not hdf5_files:
    raise FileNotFoundError(f'No HDF5 files found in {DATA_DIR}')
print(f'Found {len(hdf5_files)} HDF5 file(s): {[os.path.basename(p) for p in hdf5_files]}')

# Load the first N_EXAMPLES traces from the first file
with h5py.File(hdf5_files[0], 'r') as f:
    X_clean = np.stack(
        [f[ch][:N_EXAMPLES, :CUTOFF] for ch in INPUTS],
        axis=-1
    ).astype(np.float32)   # shape: (N_EXAMPLES, CUTOFF, n_channels)

print(f'Loaded clean signals — shape: {X_clean.shape}')
print(f'  dtype  : {X_clean.dtype}')
print(f'  range  : [{X_clean.min():.4e}, {X_clean.max():.4e}]')

## 4. Add nominal noise (`noise_const = 1.0`)

Gaussian noise is drawn from the project noise model
(`src/utils/noise.py`) independently for each channel and sample.

In [ ]:
rng = np.random.default_rng(seed=42)

X_noisy = np.zeros_like(X_clean)
for i in range(N_EXAMPLES):
    for j in range(X_clean.shape[2]):
        noise = noise_model(CUTOFF, constant=NOISE_CONST)
        X_noisy[i, :, j] = X_clean[i, :, j] + noise

print(f'Noisy signals — shape: {X_noisy.shape}')
print(f'  noise std (I ch, sample 0): {np.std(X_noisy[0, :, 0] - X_clean[0, :, 0]):.4e}')

## 5. VMD Denoising Pipeline (K = 3 modes)

VMD decomposes each 1-D signal into **K band-limited intrinsic mode functions**
(BLIMFs) that collectively reconstruct the input. Summing all K modes
recovers a filtered version of the signal with out-of-band noise suppressed.

The function `VMD(signal, alpha, tau, K, DC, init, tol)` returns:
- `u`       — modes, shape `(K, T)`
- `u_hat`   — spectral representation of modes
- `omega`   — estimated centre frequencies of each mode

In [ ]:
def vmd_denoise(signal_1d, K=K, alpha=ALPHA, tau=TAU, DC=DC, init=INIT, tol=TOL):
    """Denoise a 1-D signal using Variational Mode Decomposition.

    Parameters
    ----------
    signal_1d : np.ndarray, shape (T,)
        Input (noisy) signal.
    K : int
        Number of VMD modes.
    alpha : float
        Bandwidth constraint (larger → narrower modes).
    tau : float
        Lagrangian noise tolerance (0 = strict reconstruction).
    DC : int
        Force a DC mode (0 = no).
    init : int
        Mode initialisation strategy (1 = uniform).
    tol : float
        Convergence tolerance.

    Returns
    -------
    denoised : np.ndarray, shape (T,)
        Denoised signal (sum of all K modes).
    modes : np.ndarray, shape (K, T)
        Individual VMD modes.
    omega : np.ndarray, shape (K,)
        Estimated centre frequencies of each mode (normalised, 0–0.5).
    """
    u, u_hat, omega = VMD(signal_1d, alpha, tau, K, DC, init, tol)
    denoised = u.sum(axis=0)   # reconstruct by summing all modes
    return denoised, u, omega


# Apply VMD to all samples and channels
X_denoised = np.zeros_like(X_noisy)
all_modes  = np.zeros((N_EXAMPLES, X_clean.shape[2], K, CUTOFF), dtype=np.float32)
all_omega  = np.zeros((N_EXAMPLES, X_clean.shape[2], K),         dtype=np.float64)

for i in range(N_EXAMPLES):
    for j in range(X_clean.shape[2]):
        sig = X_noisy[i, :, j].astype(np.float64)   # VMD expects float64
        denoised, modes, omega = vmd_denoise(sig)
        X_denoised[i, :, j]     = denoised.astype(np.float32)
        all_modes[i, j]          = modes.astype(np.float32)
        all_omega[i, j]          = omega
        print(f'  Sample {i+1}, {CHANNEL_LABELS[j]}: '
              f'mode centre freqs = {omega * SAMPLING_FREQ / 1e6} MHz')

print('\nVMD denoising complete.')

## 6. Visualise individual VMD modes

Each panel shows one of the K=3 modes extracted from the noisy signal for a
single example. Mode 0 typically captures low-frequency content; mode K-1
tends to contain higher-frequency (noisier) components.

In [ ]:
SAMPLE_IDX = 0   # which sample to show
t = np.arange(CUTOFF)

for ch, ch_label in enumerate(CHANNEL_LABELS):
    fig, axes = plt.subplots(K + 1, 1, figsize=(14, 2.2 * (K + 1)), sharex=True)

    # Top panel: noisy input
    axes[0].plot(t, X_noisy[SAMPLE_IDX, :, ch], color='tab:blue',
                 lw=0.6, alpha=0.7, label='Noisy input')
    axes[0].plot(t, X_clean[SAMPLE_IDX, :, ch], color='tab:orange',
                 lw=1.2, label='Clean signal')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title(f'Sample {SAMPLE_IDX + 1} — {ch_label}: noisy input')
    axes[0].legend(loc='upper right', fontsize=8)

    # One panel per mode
    for k in range(K):
        freq_mhz = all_omega[SAMPLE_IDX, ch, k] * SAMPLING_FREQ / 1e6
        axes[k + 1].plot(t, all_modes[SAMPLE_IDX, ch, k],
                         color=f'C{k + 2}', lw=0.8)
        axes[k + 1].set_ylabel('Amplitude')
        axes[k + 1].set_title(f'Mode {k + 1}  (centre ≈ {freq_mhz:.1f} MHz)')

    axes[-1].set_xlabel('Time sample')
    fig.suptitle(f'VMD modes — {ch_label} (K={K}, α={ALPHA})', fontsize=12)
    fig.tight_layout()
    plt.show()

## 7. Denoised reconstruction vs noisy input and clean target

In [ ]:
fig, axes = plt.subplots(
    N_EXAMPLES, 2,
    figsize=(14, 2.5 * N_EXAMPLES),
    sharex=True,
)

for i in range(N_EXAMPLES):
    for ch, ch_label in enumerate(CHANNEL_LABELS):
        ax = axes[i, ch]
        ax.plot(t, X_noisy[i, :, ch],    color='tab:blue',   lw=0.5, alpha=0.5, label='Noisy')
        ax.plot(t, X_clean[i, :, ch],    color='tab:orange',  lw=1.2,            label='Clean')
        ax.plot(t, X_denoised[i, :, ch], color='tab:green',   lw=1.2, ls='--',   label='VMD denoised')
        ax.set_ylabel('Amplitude')
        ax.set_title(f'Sample {i + 1} — {ch_label}')
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('Time sample')
axes[-1, 1].set_xlabel('Time sample')
fig.suptitle(f'VMD denoising (K={K}): noisy / clean / denoised', fontsize=13)
fig.tight_layout()
plt.show()

## 8. Zoomed view

Inspect a short window to see fine-grained denoising performance.

In [ ]:
WINDOW_START = 0
WINDOW_LEN   = 512
SAMPLE_IDX   = 0

sl     = slice(WINDOW_START, WINDOW_START + WINDOW_LEN)
t_zoom = np.arange(WINDOW_START, WINDOW_START + WINDOW_LEN)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

for ch, ch_label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    ax.plot(t_zoom, X_noisy[SAMPLE_IDX, sl, ch],    color='tab:blue',  lw=0.7, alpha=0.6, label='Noisy')
    ax.plot(t_zoom, X_clean[SAMPLE_IDX, sl, ch],    color='tab:orange', lw=1.5,            label='Clean')
    ax.plot(t_zoom, X_denoised[SAMPLE_IDX, sl, ch], color='tab:green',  lw=1.5, ls='--',   label='VMD denoised')
    ax.set_xlabel('Time sample')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'Sample {SAMPLE_IDX + 1} — {ch_label} (zoom)')
    ax.legend(fontsize=8)

fig.suptitle(f'Zoomed view — samples {WINDOW_START}–{WINDOW_START + WINDOW_LEN}', fontsize=12)
fig.tight_layout()
plt.show()

## 9. Power Spectral Density comparison

Compare PSD of noisy input, clean target, and VMD-denoised signal to assess
how well VMD suppresses out-of-band noise.

In [ ]:
SAMPLE_IDX = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ch, ch_label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    for sig, name, color in [
        (X_noisy[SAMPLE_IDX, :, ch],    'Noisy input',    'tab:blue'),
        (X_clean[SAMPLE_IDX, :, ch],    'Clean target',   'tab:orange'),
        (X_denoised[SAMPLE_IDX, :, ch], 'VMD denoised',   'tab:green'),
    ]:
        f, pxx = sp_signal.periodogram(sig.astype(np.float64), fs=SAMPLING_FREQ)
        ax.semilogy(f / 1e6, pxx, lw=0.8, label=name, color=color)

    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('PSD [arb/Hz]')
    ax.set_title(f'{ch_label} — PSD')
    ax.legend(fontsize=8)

fig.suptitle('Power Spectral Density: noisy / clean / VMD-denoised', fontsize=13)
fig.tight_layout()
plt.show()

## 10. Quantitative denoising metrics

Compute per-channel and per-sample metrics:
- **MSE** — mean squared error vs clean signal  
- **SNR** — signal-to-noise ratio in dB  
- **SNR improvement** — gain over the noisy baseline

A positive SNR improvement indicates the VMD step reduces noise effectively.

In [ ]:
def snr_db(signal, reference):
    """SNR in dB: 10 * log10(power(reference) / power(signal - reference))."""
    noise_power  = np.mean((signal - reference) ** 2)
    signal_power = np.mean(reference ** 2)
    if noise_power < 1e-30:
        return np.inf
    return 10.0 * np.log10(signal_power / noise_power)


print(f'{"Sample":<8} {"Channel":<12} {"MSE (noisy)":>14} {"MSE (denoised)":>16} '
      f'{"SNR noisy (dB)":>16} {"SNR denoised (dB)":>19} {"ΔSNR (dB)":>12}')
print('-' * 100)

for i in range(N_EXAMPLES):
    for ch, ch_label in enumerate(CHANNEL_LABELS):
        clean    = X_clean[i, :, ch]
        noisy    = X_noisy[i, :, ch]
        denoised = X_denoised[i, :, ch]

        mse_noisy    = np.mean((noisy    - clean) ** 2)
        mse_denoised = np.mean((denoised - clean) ** 2)
        snr_noisy    = snr_db(noisy,    clean)
        snr_denoised = snr_db(denoised, clean)
        delta_snr    = snr_denoised - snr_noisy

        print(f'{i + 1:<8} {ch_label:<12} '
              f'{mse_noisy:>14.4e} {mse_denoised:>16.4e} '
              f'{snr_noisy:>16.2f} {snr_denoised:>19.2f} {delta_snr:>12.2f}')

## 11. Mode energy distribution

Visualise how much energy each VMD mode captures across channels and samples.
This helps diagnose whether modes cleanly separate signal from noise.

In [ ]:
mode_energies = (all_modes ** 2).mean(axis=-1)   # (N_EXAMPLES, n_channels, K)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ch, ch_label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    mean_energy = mode_energies[:, ch, :].mean(axis=0)   # average over samples
    std_energy  = mode_energies[:, ch, :].std(axis=0)
    bars = ax.bar(
        [f'Mode {k+1}' for k in range(K)],
        mean_energy,
        yerr=std_energy,
        color=[f'C{k+2}' for k in range(K)],
        capsize=5,
        edgecolor='white',
    )
    ax.set_ylabel('Mean energy (amp²)')
    ax.set_title(f'{ch_label} — mode energy')

fig.suptitle(f'VMD mode energy distribution (K={K}, error bars = std over {N_EXAMPLES} samples)', fontsize=12)
fig.tight_layout()
plt.show()

## 12. Centre-frequency summary

Print the estimated centre frequency of each VMD mode (averaged over samples)
for both I and Q channels.

In [ ]:
print('VMD mode centre frequencies (averaged over all samples):')
print(f'{"Mode":<8} {"I channel (MHz)":>18} {"Q channel (MHz)":>18}')
print('-' * 46)

mean_omega = all_omega.mean(axis=0)   # (n_channels, K)
for k in range(K):
    freq_I = mean_omega[0, k] * SAMPLING_FREQ / 1e6
    freq_Q = mean_omega[1, k] * SAMPLING_FREQ / 1e6
    print(f'{k + 1:<8} {freq_I:>18.2f} {freq_Q:>18.2f}')